# UNIFAC

A model for liquid activity coefficients is a must for vapour-liquid equilibrium. Luckely, the UNIFAC model is a group-contribution model compatible with the Constantinou-Gani model e.i. they both use the same way of describing the molecule.

The reader is asked to check the following references:
* Introduction to chemical engineering thermodynamics
* [Model-based solvent selection for the synthesis and crystallisation of pharmaceutical compounds](https://www.sciencedirect.com/science/article/pii/S0009250922007096)

UNIFAC Activity Coefficient Model

Total Activity Coefficient:
\begin{equation*}
\gamma_i = \gamma_i^C \cdot \gamma_i^R
\end{equation*}

Logarithmic Form:
\begin{equation*}
\ln \gamma_i = \ln \gamma_i^C + \ln \gamma_i^R
\end{equation*}

## Combinatorial Part (Size and Shape Effects)

Volume and Surface Area Fractions:
\begin{equation*}
r_i = \sum_k \nu_{i,k} R_k
\end{equation*}

\begin{equation*}
q_i = \sum_k \nu_{i,k} Q_k
\end{equation*}

Segment and Area Fractions:
\begin{equation*}
J_i = \frac{r_i}{\sum_j x_j r_j}
\end{equation*}

\begin{equation*}
L_i = \frac{q_i}{\sum_j x_j q_j}
\end{equation*}

Combinatorial Activity Coefficient:
\begin{equation*}
\ln \gamma_i^C = 1 - J_i + \ln J_i - 5q_i\left(1 - \frac{J_i}{L_i} + \ln \frac{J_i}{L_i}\right)
\end{equation*}

## Residual Part (Group Interaction Effects)

Group Surface Area Fraction in Component i:
\begin{equation*}
e_{i,k} = \frac{\nu_{i,k} Q_k}{q_i}
\end{equation*}

Group Surface Area Fraction in Mixture:
\begin{equation*}
\theta_k = \frac{\sum_j x_j \nu_{j,k} Q_k}{\sum_j x_j q_j}
\end{equation*}

Group Interaction Parameter:
\begin{equation*}
\psi_{mn} = \exp\left(-\frac{a_{mn}}{T}\right)
\end{equation*}

Group Activity Coefficient in Component i:
\begin{equation*}
\beta_{i,m} = \frac{\sum_k \nu_{i,k} Q_k \psi_{km}}{q_i}
\end{equation*}

Group Activity Coefficient in Mixture:
\begin{equation*}
\omega_m = \sum_k \theta_k \psi_{km}
\end{equation*}

Residual Activity Coefficient:
\begin{equation*}
\ln \gamma_i^R = q_i \sum_k \left(\theta_k \frac{\beta_{i,k}}{\omega_k} - e_{i,k} \ln \frac{\beta_{i,k}}{\omega_k}\right)
\end{equation*}

The UNIFAC implementation is validated by producing the liquid activity coeﬃcients for a binary mixture of Diethylamine (DEA) and n-heptane (HEP) at 308.15 K with the compoisitions being 0.4 and 0.6 respectively. This example is reported in the book mentioned above and the fila results should be:
* 

In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np

# import the new UNIFAC interface
from src.data import load_unifac_data
from src.mixture.unifac import unifac

print("Loading UNIFAC data...")
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()

# Define the mixture in standard compound format
mixture = {
    'heptane': {'CH3': 2, 'CH2': 5},
    'DEA': {'CH3': 2, 'CH2': 1, 'CH2NH': 1}
}

# Define compositions and temperature as numpy arrays
T = np.array([308.15])  # K, shape: (1,)
x = np.array([[0.6, 0.4]])  # 0.6 heptane, 0.4 DEA, shape: (1, 2)


# Calculate UNIFAC activity coefficients
results = unifac(mixture, x, T, rk_data, qk_data, interaction_data) 

print(f"\nResults:")
print(f"  γ array shape: {results['gamma'].shape}")  # Should be (1, 2)
print(f"  γ_heptane = {results['gamma'][0,0]:.4f}")
print(f"  γ_DEA = {results['gamma'][0,1]:.4f}")

we might be intrested in evaluating multiple compositions at corresponding temperatures. Below is an example

In [ ]:
# lets consider the following example
# define number of data points
n_data = 10
# set tht temperature
T = np.full(n_data, 308.15) 
# set composition
x1 = np.linspace(0, 1, n_data)  # Component 1: 0 to 1
x2 = 1 - x1                     # Component 2: 1 to 0 (complement)

x = np.column_stack([x1, x2])  # Combine into 2D array

# Same mixture format
results = unifac(mixture, x, T, rk_data, qk_data, interaction_data)

print(f"\nResults:")
print(f"  γ array shape: {results['gamma'].shape}")  # Should be (3, 2)
print(f"  Activity coefficients:")
for i, (T_i, x_i) in enumerate(zip(T, x)):
    print(f"    Point {i+1}: T={T_i:.1f}K, x=[{x_i[0]:.1f}, {x_i[1]:.1f}] -> γ=[{results['gamma'][i,0]:.4f}, {results['gamma'][i,1]:.4f}]")

if we plot this we can validate the Gibbs Duhem where the slope should approach zero.


In [ ]:
import matplotlib.pyplot as plt

 # Create the plot                                                                                                            
plt.figure(figsize=(6, 6))                                                                                                                     
plt.plot(x1, results['gamma'][:, 0], 'b-', linewidth=2, label='γ_heptane', marker='o', markersize=4)                                         
plt.plot(x1, results['gamma'][:, 1], 'r-', linewidth=2, label='γ_DEA', marker='s', markersize=4)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ideal solution (γ = 1)')

plt.xlabel('Mole fraction heptane (x_heptane)', fontsize=12)
plt.ylabel('Activity coefficient (γ)', fontsize=12)
plt.title(f'UNIFAC Activity Coefficients: Heptane-DEA System at {T[0]:.0f} K', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.ylim(bottom=0.8)

# Add some annotation
plt.text(0.05, results['gamma'][0, 1]*0.95, f"γ_DEA → {results['gamma'][0, 1]:.2f}\n(x_DEA → 1)",
           fontsize=10, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3',
  facecolor='lightcoral', alpha=0.7))
plt.text(0.95, results['gamma'][-1, 0]*0.95, f"γ_heptane → {results['gamma'][-1, 0]:.2f}\n(x_heptane → 1)",
           fontsize=10, ha='right', va='top', bbox=dict(boxstyle='round,pad=0.3',
  facecolor='lightblue', alpha=0.7))
plt.tight_layout()
plt.show()

## UNIFAC-POYOMO

While ti is nice to have a numpy implementation of unifac, we need to integrate unifac in pyomo.

So here we show how to perform the examt same calculations just using pyomo-unifac.

In [ ]:
from src.mixture.unifac.pyomo_unifac import pyo_unifac, _convert_mixture_format

_convert_mixture_format(mixture)
sol, model = pyo_unifac(x, T, mixture, rk_data, qk_data, interaction_data)#
# retrieving the solutions
gamma_pyo = np.empty_like(x)
T_pyo = np.empty_like(T)


for i in model.gamma.keys():
    gamma_pyo[i] = model.gamma[i].value

for i in model.T.keys():
    T_pyo[i] = model.T[i]
gamma_pyo   

In [ ]:
import matplotlib.pyplot as plt

 # Create the plot                                                                                                            
plt.figure(figsize=(6, 6))                                                                                                                     
plt.plot(x1, gamma_pyo[:, 0], 'b-', linewidth=2, label='γ_heptane', marker='o', markersize=4)                                         
plt.plot(x1, gamma_pyo[:, 1], 'r-', linewidth=2, label='γ_DEA', marker='s', markersize=4)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ideal solution (γ = 1)')

plt.xlabel('Mole fraction heptane (x_heptane)', fontsize=12)
plt.ylabel('Activity coefficient (γ)', fontsize=12)
plt.title(f'UNIFAC Activity Coefficients: Heptane-DEA System at {T[0]:.0f} K', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.ylim(bottom=0.8)

# Add some annotation
plt.text(0.05, gamma_pyo[0, 1]*0.95, f"γ_DEA → {gamma_pyo[0, 1]:.2f}\n(x_DEA → 1)",
           fontsize=10, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3',
  facecolor='lightcoral', alpha=0.7))
plt.text(0.95, gamma_pyo[-1, 0]*0.95, f"γ_heptane → {gamma_pyo[-1, 0]:.2f}\n(x_heptane → 1)",
           fontsize=10, ha='right', va='top', bbox=dict(boxstyle='round,pad=0.3',
  facecolor='lightblue', alpha=0.7))
plt.tight_layout()
plt.show()

## Pyomo vs Interface Comparison

Let's compare the results from the Pyomo model with the numerical interface to validate the implementation.

In [ ]:
# Calculate results from both methods
from src.mixture.unifac.pyomo_unifac import pyo_unifac
import pyomo.environ as pyo

# Use single point for comparison
x_single = np.array([[0.6, 0.4]])
T_single = np.array([308.15])

print("Calculating Interface (numerical) results...")
results_interface = unifac(mixture, x_single, T_single, rk_data, qk_data, interaction_data)

print("Calculating Pyomo (symbolic) results...")
solution_pyo, model_pyo = pyo_unifac(x_single, T_single, mixture, rk_data, qk_data, interaction_data)

print("✓ Both calculations completed")

In [ ]:
# Compare r values (molecular volume parameters)
print("=== r values (molecular volume parameters) ===")
r_interface = results_interface['r']  # Extract first (and only) data point
r_pyomo = [pyo.value(model_pyo.r[i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(r_interface)):
    diff = abs(r_interface[i] - r_pyomo[i])
    print(f"    {i}     | {r_interface[i]:8.4f}  | {r_pyomo[i]:8.4f}  | {diff:8.6f}")

print(f"\nMax difference: {max(abs(r_interface[i] - r_pyomo[i]) for i in range(len(r_interface))):.2e}")

In [ ]:
# Compare q values (molecular surface area parameters)
print("=== q values (molecular surface area parameters) ===")
q_interface = results_interface['q']
q_pyomo = [pyo.value(model_pyo.q[i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(q_interface)):
    diff = abs(q_interface[i] - q_pyomo[i])
    print(f"    {i}     | {q_interface[i]:8.4f}  | {q_pyomo[i]:8.4f}  | {diff:8.6f}")

print(f"\nMax difference: {max(abs(q_interface[i] - q_pyomo[i]) for i in range(len(q_interface))):.2e}")

In [ ]:
# Compare J and L values (volume and surface fractions)
print("=== J values (volume fractions) ===")
J_interface = results_interface['J']
J_pyomo = [pyo.value(model_pyo.J[0, i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(J_interface)):
    diff = abs(J_interface[i] - J_pyomo[i])
    print(f"    {i}     | {J_interface[i]:8.4f}  | {J_pyomo[i]:8.4f}  | {diff:8.6f}")

print("\\n=== L values (surface fractions) ===")
L_interface = results_interface['L']
L_pyomo = [pyo.value(model_pyo.L[0, i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(L_interface)):
    diff = abs(L_interface[i] - L_pyomo[i])
    print(f"    {i}     | {L_interface[i]:8.4f}  | {L_pyomo[i]:8.4f}  | {diff:8.6f}")

In [ ]:
# Compare e values (group surface fractions in components)
print("=== e values (group surface fractions in components) ===")
e_interface = results_interface['e']  # Shape: (n_components, n_groups)
groups = list(mixture_groups.keys()) if 'mixture_groups' in locals() else ['CH2', 'CH2NH', 'CH3']

print("Component | Group     | Interface | Pyomo     | Difference")
print("-" * 55)
for i in range(e_interface.shape[0]):  # Components
    for j, group in enumerate(groups):  # Groups
        if j < e_interface.shape[1]:
            e_int = e_interface[i, j]
            e_pyo = pyo.value(model_pyo.e[i, group])
            diff = abs(e_int - e_pyo)
            print(f"    {i}     | {group:8s}  | {e_int:8.4f}  | {e_pyo:8.4f}  | {diff:8.6f}")

max_diff_e = max(abs(e_interface[i, j] - pyo.value(model_pyo.e[i, groups[j]])) 
                for i in range(e_interface.shape[0]) for j in range(min(len(groups), e_interface.shape[1])))
print(f"\\nMax difference: {max_diff_e:.2e}")

In [ ]:
# Compare theta values (group surface area fractions in mixture)
print("=== theta values (group surface area fractions in mixture) ===")
theta_interface = results_interface['theta'][0]  # Shape: (n_groups,)

print("Group     | Interface | Pyomo     | Difference")
print("-" * 45)
for j, group in enumerate(groups):
    if j < len(theta_interface):
        theta_int = theta_interface[j]
        theta_pyo = pyo.value(model_pyo.theta[0, group])
        diff = abs(theta_int - theta_pyo)
        print(f"{group:8s}  | {theta_int:8.4f}  | {theta_pyo:8.4f}  | {diff:8.6f}")

max_diff_theta = max(abs(theta_interface[j] - pyo.value(model_pyo.theta[0, groups[j]])) 
                    for j in range(min(len(groups), len(theta_interface))))
print(f"\\nMax difference: {max_diff_theta:.2e}")

In [ ]:
# Compare psi values (group interaction parameters)
print("=== psi values (group interaction parameters) ===")
psi_interface = results_interface['psi']  # Shape: (n_groups, n_groups)

print("Group i   | Group j   | Interface | Pyomo     | Difference")
print("-" * 60)
for i, group_i in enumerate(groups):
    for j, group_j in enumerate(groups):
        if i < psi_interface.shape[0] and j < psi_interface.shape[1]:
            psi_int = psi_interface[i, j]
            psi_pyo = pyo.value(model_pyo.psi[0, group_i, group_j])  # tau = psi in Pyomo
            diff = abs(psi_int - psi_pyo)
            print(f"{group_i:8s}  | {group_j:8s}  | {psi_int:8.4f}  | {psi_pyo:8.4f}  | {diff:8.6f}")

max_diff_psi = max(abs(psi_interface[i, j] - pyo.value(model_pyo.psi[0, groups[i], groups[j]])) 
                  for i in range(min(len(groups), psi_interface.shape[0])) 
                  for j in range(min(len(groups), psi_interface.shape[1])))
print(f"\\nMax difference: {max_diff_psi:.2e}")

In [ ]:
# Compare beta values (group interaction parameters)
print("=== beta values (group interaction parameters) ===")
beta_interface = results_interface['beta']  # Shape: (n_data_points, n_components, n_groups)

print("Component | Group     | Interface | Pyomo     | Difference")
print("-" * 60)
for i in range(beta_interface.shape[0]):  # component index
    for k in range(beta_interface.shape[1]):  # group index
        beta_int = beta_interface[i, k]  # First data point
        beta_pyo = pyo.value(model_pyo.beta[0, i, groups[k]])
        diff = abs(beta_int - beta_pyo)
        comp_name = list(mixture.keys())[i] if i < len(mixture) else f"comp_{i}"
        group_name = groups[k]
        print(f"{comp_name:8s}  | {group_name:8s}  | {beta_int:8.4f}  | {beta_pyo:8.4f}  | {diff:8.6f}")

In [ ]:
# Compare omega values (omega parameters from mixture)
print("=== omega values (omega parameters) ===")
omega_interface = results_interface['omega'][0]  # Shape: (n_groups,)

print("Group     | Interface | Pyomo     | Difference")
print("-" * 45)
for j, group in enumerate(groups):
    if j < len(omega_interface):
        omega_int = omega_interface[j]
        omega_pyo = pyo.value(model_pyo.omega[0, group])  # eta = omega in Pyomo
        diff = abs(omega_int - omega_pyo)
        print(f"{group:8s}  | {omega_int:8.4f}  | {omega_pyo:8.4f}  | {diff:8.6f}")

max_diff_omega = max(abs(omega_interface[j] - pyo.value(model_pyo.omega[0, groups[j]])) 
                    for j in range(min(len(groups), len(omega_interface))))
print(f"\\nMax difference: {max_diff_omega:.2e}")

print("\\n" + "="*60)
print("Summary: All intermediate variables have been compared!")
print("Small differences (< 1e-6) are typically due to numerical precision.")
print("="*60)

In [ ]:
# Compare combinatorial and residual activity coefficients
print("=== ln(γ^C) values (combinatorial) ===")
ln_gamma_c_interface = results_interface['log_gamma_c']
ln_gamma_c_pyomo = [pyo.value(model_pyo.log_gamma_c[0, i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(ln_gamma_c_interface)):
    diff = abs(ln_gamma_c_interface[i] - ln_gamma_c_pyomo[i])
    print(f"    {i}     | {ln_gamma_c_interface[i]:8.4f}  | {ln_gamma_c_pyomo[i]:8.4f}  | {diff:8.6f}")

print("\\n=== ln(γ^R) values (residual) ===")
ln_gamma_r_interface = results_interface['log_gamma_r']
ln_gamma_r_pyomo = [pyo.value(model_pyo.log_gamma_r[0, i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference")
print("-" * 45)
for i in range(len(ln_gamma_r_interface)):
    diff = abs(ln_gamma_r_interface[i] - ln_gamma_r_pyomo[i])
    print(f"    {i}     | {ln_gamma_r_interface[i]:8.4f}  | {ln_gamma_r_pyomo[i]:8.4f}  | {diff:8.6f}")

In [ ]:
# Compare final activity coefficients
print("=== γ values (total activity coefficients) ===")
gamma_interface = results_interface['gamma'][0]
gamma_pyomo = [pyo.value(model_pyo.gamma[0, i]) for i in model_pyo.i]

print("Component | Interface | Pyomo     | Difference | Rel. Error")
print("-" * 60)
for i in range(len(gamma_interface)):
    diff = abs(gamma_interface[i] - gamma_pyomo[i])
    rel_error = diff / gamma_interface[i] * 100
    print(f"    {i}     | {gamma_interface[i]:8.4f}  | {gamma_pyomo[i]:8.4f}  | {diff:8.6f} | {rel_error:6.3f}%")

max_diff = max(abs(gamma_interface[i] - gamma_pyomo[i]) for i in range(len(gamma_interface)))
max_rel_error = max(abs(gamma_interface[i] - gamma_pyomo[i])/gamma_interface[i] for i in range(len(gamma_interface))) * 100

print(f"\\nMax absolute difference: {max_diff:.2e}")
print(f"Max relative error: {max_rel_error:.3f}%")

if max_rel_error < 0.01:
    print("✅ Excellent agreement between methods!")
elif max_rel_error < 0.1:
    print("✅ Good agreement between methods!")
else:
    print("⚠️  Check for implementation differences")